In [1]:
import pandas as pd
import os
import math
import numpy as np
import json
import random
import re
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from tqdm import tqdm
import datetime

In [301]:
os.chdir('/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis')

In [302]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'

In [304]:
nace_description_path = "data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv"
nace_descriptions = pd.read_csv(nace_description_path, sep="\t")

In [305]:
def get_zero_shot_user_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["user_prompt_zero_shot"]

def get_few_shot_user_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["user_prompt_few_shot"]

def get_system_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["system_prompt"]

In [306]:
user_prompt_topic = """
TASK
For the following business sector, create short descriptions of realistic business models for exisiting companies. 

DEFINITION
{includes} {includes_also}

{excludes}

INSTRUCTION
- Create a list of {num_samples} different realistic explanations of a business model 
- The length should be one sentence
- Pick one or multiple subsections for this business model
- Do not explain what you did or what you used
"""

In [307]:
def generate_topic(
        num_samples: int, 
        includes: str,
        includes_also: str, 
        excludes: str,
        prompt_path: str,  
        temperature: float = 0.4, 
        model: str = "gpt-4o-mini"
): 

    # Initialize LLM
    if model == "gpt-4o-mini":
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )
    elif model == "openai/gpt-oss-120b":
        llm = ChatOllama(
            model=model,
            temperature=temperature, 
            base_url="http://10.80.20.127:11434/"
        )


    prompt = ChatPromptTemplate.from_messages([
        ("system", ""),
        ("human", user_prompt_topic)
    ])

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        }

    formatted_prompt = prompt.invoke(input)

    # print("Formatted User Prompt:", formatted_prompt.messages[1].content)

    # Run
    response = chain.invoke(input)

    # print(response.content)

    return formatted_prompt, response.content

In [308]:
def generate_synthetic_data(
        num_samples: int, 
        gold_standard: list, 
        includes: str,
        includes_also: str, 
        excludes: str,
        subsections: list,
        prompt_path: str,  
        temperature: float = 0.4, 
        model: str = "gpt-4o-mini", 
        topic: str = None
): 

    # Initialize LLM
    if model == "gpt-4o-mini":
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )
    elif model == "openai/gpt-oss-120b":
        llm = ChatOllama(
            model=model,
            temperature=temperature, 
            base_url="http://10.80.20.127:11434/"
        )

    # Prompt
    if gold_standard == []: 
        #print("Zero Shot!")
        prompt = ChatPromptTemplate.from_messages([
            ("system", get_system_prompt(prompt_path)),
            ("human", get_zero_shot_user_prompt(prompt_path))
        ])
        gold_standard_str = ""
        
    else: 
        prompt = ChatPromptTemplate.from_messages([
            ("system", get_system_prompt(prompt_path)),
            ("human", get_few_shot_user_prompt(prompt_path))
        ])
        gold_standard_str = ""
        for i, text in enumerate(gold_standard): 
            gold_standard_str += f"Example {i+1}:\n{text}\n\n"
        gold_standard_str = gold_standard_str[:-2]

    # Subsections string
    subsections_str = "\n - " + "\n - ".join(subsections)

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "gold_standard": gold_standard_str,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        "subsections": subsections_str,
        }

    # is topic used? topic in params -> business model must be in user prompt
    if topic is not None:
        if "business_model" not in prompt.input_variables: 
            print("If topic is given, prompt must conatin '{business_model}'")
            return False
        else: 
            if isinstance(topic, list): 
                topic_str = "\n".join([f"{i+1}: {text}" for i, text in enumerate(topic)])
                input["business_model"] = topic_str
            else:
                input["business_model"] = topic

    formatted_prompt = prompt.invoke(input)

    #print("Formatted Prompt:", formatted_prompt)

    # Run
    response = chain.invoke(input)

    #print(response.content)

    return formatted_prompt, response.content

In [309]:
def split_synthetic_data(content: str, num_samples: int): 
    content_list = content.split("\n")
    content_list = [c for c in content_list if c != ""]
    # if len(content_list) != num_samples: 
    #     print("Warning: length of creates examples != num_samples!")
    return content_list

In [310]:
def get_sublevels(nace_class, level): 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(f'{row["NAME"]}')
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(f'{row["NAME"]}: {row_2["NAME"]}')
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(f'{row["NAME"]}: {row_2["NAME"]}: {row_3["NAME"]}')
        
    if level == 2: 
        return nace_class_lvl_2
    if level == 3: 
        return nace_class_lvl_3
    if level == 4: 
        return nace_class_lvl_4

In [311]:
def get_sublevels_df(nace_class, level) -> pd.DataFrame: 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(row)
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(row_2)
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(row_3)
        
    if level == 2: 
        df =  pd.concat(nace_class_lvl_2, axis=1).T
        return df.drop_duplicates()

    if level == 3: 
        df = pd.concat(nace_class_lvl_3, axis=1).T
        return df.drop_duplicates()

    if level == 4: 
        df = pd.concat(nace_class_lvl_4, axis=1).T
        return df.drop_duplicates()

In [312]:
generate_nace_class = "A"

includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
assert includes is not None and includes != ""
includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
includes_also = "" if pd.isna(includes_also) else includes_also
excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
excludes = "" if pd.isna(excludes) else excludes

gold_standard = []

## Generate Few-Shot Data

In [313]:
# select gold standard data

# ds_2_desc  = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

# ds_2_desc  = ds_2_desc[pd.notna(ds_2_desc["Description"])]

# gold_standard = []
# # 1. take one of each lvl 3 class:
# for lvl_3 in ds_2_desc[pd.notna(ds_2_desc["Description"])].groupby("NACE_lvl_3").size().index: 
#     gold_standard.append(ds_2_desc[ds_2_desc["NACE_lvl_3"] == lvl_3].iloc[0])

# df_gold_standard = pd.concat(gold_standard, axis=1).T

#df_gold_standard.to_csv("/Users/hendrikweichel/Downloads/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv")

In [314]:
#df_gold_standard = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")
df_gold_standard = pd.read_csv("data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

### Hyperparams

In [315]:
prompt_path = "generate_synthetic_data/prompts/prompts_6.json"
print("System Prompt: ", get_system_prompt(prompt_path))
print("User Prompt: ", get_few_shot_user_prompt(prompt_path))

System Prompt:  You generate realistic paragraphs from corporate annual reports that are a presentation of the copmanies business model. The text must: - sound natural and specific - avoid textbook or definitional language - vary structure, length, and narrative style - include concrete operational details The text must NOT: - repeat industry definitions - follow a fixed template - explicitly explain what the company does in generic terms
User Prompt:  
TASK 
You get list of short descriptions of a companies' business models. Use these descriptions to formulate it into a typical description within an annual report.

Here are some examples of descriptions of these classes: 
```
{gold_standard}
```

BUSINESS MODELS
{business_model}

INSTRUCTIONS
 - For each given Business Model, create a business model description that is in a similar format as the examples
 - Do NOT exactly copy phrases, sentence patterns, or structure from the examples
 - Think of the examples as constraints, not templ

In [316]:
level = 1
head_nace_code = "1" if level > 1 else None
generated_classes = nace_descriptions[nace_descriptions["PARENT_ID"] == head_nace_code]["CODE"]
few_shot = True

In [317]:
# generate date 

date = datetime.datetime.now().strftime("%Y%m%d")
date = "20251218"
suffix = "__few_shot" if few_shot else "__zero_shot"
suffix += "__from_lvl_4_topics"
store_path = "data/synthetic_data/data_" + date + f"__level_{level}__subclasses_{head_nace_code}__{os.path.basename(prompt_path).replace('.json', '')}{suffix}/" 
os.makedirs(store_path, exist_ok=True)

In [318]:
generated_data = {}

In [319]:
generated_classes = ["A", "B", "C", "J", "F"]
generated_classes = ["A", "C"]

In [320]:
# # load previous results
# prev_results = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot/class_A.csv"
# prev_results = "data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot"
# for class_name in generated_classes: 
#    file_path = os.path.join(prev_results, f"class_{class_name}.csv")
#    try:
#        df = pd.read_csv(file_path)
#        generated_data[class_name] = {
#            "data": df[class_name].tolist()
#        }
#    except Exception as e:
#        print(f"Could not load previous results for class {class_name}: {e}")

### Generate Topics

Here: 

- for each class get the subclasses on level 4
- for each subclass: 
    - generate k topics such that n_data topics are created

In [321]:
n_data = 500
num_samples = 20
iterations_ = n_data // num_samples

In [ ]:
topics = {}

for generate_nace_class in generated_classes:
#for generate_nace_class in ["C"]:

    df_subsections = get_sublevels_df(generate_nace_class, level=4)

    topics_per_subsec = math.ceil(n_data / 5)
    topic_samples = min(topics_per_subsec, 50)
    topic_iterations = math.ceil(topics_per_subsec / topic_samples)

    examples = []

    for i, subsection in df_subsections.iterrows(): 

        includes = subsection["Includes"]
        if pd.isna(includes):
            includes = subsection["NAME"]

        includes_also = subsection["IncludesAlso"]
        includes_also = "" if pd.isna(includes_also) else includes_also
        excludes = subsection["Excludes"]
        excludes = "" if pd.isna(excludes) else excludes

        for i in tqdm(range(topic_iterations), desc=generate_nace_class):
            res = generate_topic(num_samples=topic_samples, 
                                includes=includes, 
                                includes_also=includes_also, 
                                excludes=excludes,  
                                prompt_path=prompt_path, 
                                model="gpt-4o-mini", 
                                temperature=0.8)
            examples.append(res[1])
            data = split_synthetic_data("\n\n".join(examples), num_samples * iterations_)

    results = {
        "topics": random.sample(data, k=n_data),
#        "prompt": res[0],
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": examples,
        "few_shot": few_shot
    }

    topics[generate_nace_class] = results

C:  50%|████████████████████████████████████████████████████████████████████████████████████████                                                                                        | 1/2 [00:26<00:26, 26.22s/it]

In [ ]:
clean_text = lambda x: re.sub(r'^\d+\.\s*', " ", x).strip()

for class_ in generated_classes:
    topics[class_]["topics"] = list(map(clean_text, topics["A"]["topics"]))

In [ ]:
for e in examples: 
    print("---")
    print(e)

---
1. A meat processing company that sources locally raised livestock and produces artisan sausages for gourmet markets and restaurants.

2. A fish processing plant that specializes in sustainable seafood products, selling frozen and canned fish to retail chains and food service providers.

3. A fruit and vegetable processing facility that creates organic juice blends and smoothies, focusing on health-conscious consumers.

4. A dairy cooperative that transforms raw milk into a variety of cheese products, selling directly to consumers through a subscription service.

5. A bakery that produces gluten-free and vegan bread products, distributing to health food stores and cafes across the region.

6. An animal feed manufacturer that uses by-products from grain milling to create nutritious, affordable feed for livestock and poultry.

7. A custom slaughtering business that provides processing services for local farmers, returning packaged meat directly to the producers.

8. A grain mill that

In [ ]:
with open(os.path.join(store_path, "topics.json"), "w") as f: 
    json.dump(topics, f, indent=4)

### Generate Descriptions

In [ ]:
generated_data = {}

In [ ]:
for generate_nace_class in generated_classes:

    includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
    if pd.isna(includes):
        print("No description for class:", generate_nace_class)
        continue
    includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
    includes_also = "" if pd.isna(includes_also) else includes_also
    excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
    excludes = "" if pd.isna(excludes) else excludes

    if few_shot: 
        gold_standard = df_gold_standard[df_gold_standard["NACE_letter"] == generate_nace_class]["Description_clean"].to_list()[:3] 
        gold_standard = [text.replace("\n", "") for text in gold_standard]
    else: 
        gold_standard = [] 

    subsections = get_sublevels(generate_nace_class, level=2)

    examples = []

    if generated_data.get(generate_nace_class) is not None:
        if len(generated_data[generate_nace_class].get("data", [])) > 0:
            examples = generated_data[generate_nace_class]["data"]

    for i in tqdm(range(iterations_), desc=generate_nace_class):

        iteration_topics = topics[generate_nace_class]["topics"][num_samples*i:num_samples*(i+1)]
        res = generate_synthetic_data(num_samples=num_samples, gold_standard=gold_standard, includes=includes, includes_also=includes_also, excludes=excludes, subsections=subsections, prompt_path=prompt_path, model="gpt-4o-mini", topic=iteration_topics)
        examples.append(res[1])
        data = split_synthetic_data("\n\n".join(examples), num_samples * iterations_)
        pd.DataFrame(data, columns=[generate_nace_class]).to_csv(os.path.join(store_path, f"class_{generate_nace_class}.csv"), index=False)

        if i < 2: 
            [print(example) for example in examples]
            print(res[0].messages[1].content)

        if len(data) >= num_samples * iterations_:
            break

    results = {
        "data": data,
        "prompt": res[0],
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": examples, 
        "few_shot": few_shot
    }

    generated_data[generate_nace_class] = results

A:   0%|                                                                                                                                                                                       | 0/25 [00:00<?, ?it/s]

A:   4%|███████                                                                                                                                                                        | 1/25 [00:37<15:07, 37.82s/it]

1. Our company operates a dynamic tech platform that bridges the gap between local wood suppliers and artisans in search of quality raw materials. By streamlining the procurement process, we empower craftsmen to access sustainably sourced timber with ease. This innovative approach not only enhances operational efficiency but also fosters a community of local suppliers, ensuring that artisans can create their masterpieces while supporting regional economies.

2. We provide expert consulting services tailored to landowners aiming to maximize the potential of their forests. Our team specializes in guiding clients through the intricacies of managing timber and non-timber product extraction, ensuring that economic viability aligns with environmental stewardship. By offering tailored strategies, we help landowners cultivate thriving ecosystems that yield both financial returns and biodiversity.

3. Nestled in a picturesque setting, our organic dairy farm is dedicated to crafting artisanal ch

A:   8%|██████████████                                                                                                                                                                 | 2/25 [01:16<14:36, 38.09s/it]

1. Our company operates a dynamic tech platform that bridges the gap between local wood suppliers and artisans in search of quality raw materials. By streamlining the procurement process, we empower craftsmen to access sustainably sourced timber with ease. This innovative approach not only enhances operational efficiency but also fosters a community of local suppliers, ensuring that artisans can create their masterpieces while supporting regional economies.

2. We provide expert consulting services tailored to landowners aiming to maximize the potential of their forests. Our team specializes in guiding clients through the intricacies of managing timber and non-timber product extraction, ensuring that economic viability aligns with environmental stewardship. By offering tailored strategies, we help landowners cultivate thriving ecosystems that yield both financial returns and biodiversity.

3. Nestled in a picturesque setting, our organic dairy farm is dedicated to crafting artisanal ch

C:   4%|███████                                                                                                                                                                        | 1/25 [00:29<11:53, 29.72s/it]

1: Our innovative technology platform serves as a vital link between local wood suppliers and artisans in search of high-quality raw materials. By streamlining the procurement process, we empower artisans to access sustainably sourced timber, enhancing their creative projects while supporting local economies. Our user-friendly interface facilitates seamless transactions, allowing artisans to focus on their craft while we manage logistics and supplier relationships. This approach not only fosters a thriving community of creators but also promotes sustainable forestry practices that benefit the environment.

2: We specialize in providing expert consulting services tailored for landowners seeking to optimize their forest management practices. Our team of seasoned professionals collaborates closely with clients to develop customized strategies that balance timber production with the sustainable extraction of non-timber products. By leveraging our deep understanding of forestry ecosystems, 

C:   8%|██████████████                                                                                                                                                                 | 2/25 [00:56<10:50, 28.28s/it]

1: Our innovative technology platform serves as a vital link between local wood suppliers and artisans in search of high-quality raw materials. By streamlining the procurement process, we empower artisans to access sustainably sourced timber, enhancing their creative projects while supporting local economies. Our user-friendly interface facilitates seamless transactions, allowing artisans to focus on their craft while we manage logistics and supplier relationships. This approach not only fosters a thriving community of creators but also promotes sustainable forestry practices that benefit the environment.

2: We specialize in providing expert consulting services tailored for landowners seeking to optimize their forest management practices. Our team of seasoned professionals collaborates closely with clients to develop customized strategies that balance timber production with the sustainable extraction of non-timber products. By leveraging our deep understanding of forestry ecosystems, 

C:  96%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████       | 24/25 [13:37<00:34, 34.07s/it]


In [ ]:
print(res[0].messages[1].content)

TASK 
You get list of short descriptions of a companies' business models. Use these descriptions to formulate it into a typical description within an annual report.

DEFINITION
This section includes the physical or chemical transformation of materials, substances, or components into new products, although this cannot be used as the single universal criterion for defining manufacturing (see remark on processing of waste below). The materials, substances, or components transformed are raw materials that are products of agriculture, forestry, fishing, mining or quarrying as well as products of other manufacturing activities. Substantial alteration, renovation or reconstruction of goods is generally considered to be manufacturing.\n\nThe output of a manufacturing process may be finished in the sense that it is ready for utilisation or consumption, or it may be semi-finished in the sense that it is to become an input for further manufacturing. For example, the output of alumina refining is 

In [ ]:
# print prompts
for k,v in generated_data.items(): 
    print(k,len(v["output"]), "_______"*20)
    for k in v["output"]: 
        print(k)
        print("_")

A 20 ____________________________________________________________________________________________________________________________________________
The company operates a pioneering rice farm that champions sustainable agricultural practices, prioritizing eco-friendly methods to cultivate high-quality organic rice. With a strong commitment to environmental stewardship, we employ innovative techniques that enhance soil health and biodiversity, ensuring our farming operations have a minimal ecological footprint. Our rice is grown in harmony with nature, utilizing natural pest control and organic fertilizers that contribute to the overall vitality of the ecosystem.

In addition to our cultivation practices, we emphasize local distribution channels, connecting directly with health-conscious consumers who value transparency and quality in their food sources. By fostering relationships with local markets and retailers, we ensure that our organic rice reaches customers at peak freshness, reinfo

In [ ]:
# print prompts
for k,v in generated_data.items(): 
    print(k,"_______"*20)
    print(v["user_prompt"])


A ____________________________________________________________________________________________________________________________________________

TASK 
You get a short description of a companies' busines model and rephrase it such that it fits into a typical description within an annual report.

DEFINITION
This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. 



Here are some examples of descriptions of these classes: 
```
Example 1:
The company, together with its subsidiaries, operates as one of the largest vertically integrated agricultural groups in Ukraine, engaging in the production, storage, processing, and sale of agricultural products. The company's key activities include breeding pigs, processing pork, and producing wheat and sunflower. The Group focuses on three winter 

#### Aggregate data and split

In [ ]:
# config

config = {
    "prompts": {k: v.get("user_prompt") for k, v in generated_data.items()}, 
    #"samples": num_samples * iterations_,
    #"generated_iterations": iterations_,
    "level": level,
    "head_nace_code": head_nace_code,
    "system_prompt": get_system_prompt(prompt_path), 
    "few_shot_prompting": few_shot
}

# store
import json
with open(os.path.join(store_path, "config.json"), "w") as f: 
    json.dump(config, f, indent=4)

In [ ]:
df_full = []
for k, v in generated_data.items(): 
    df_temp = pd.DataFrame(v["data"], columns=["text"])
    df_temp["label"] = k
    df_full.append(df_temp)
df_full = pd.concat(df_full, axis=0)
df_full = df_full.reset_index(drop=True)


In [ ]:
import re
clean_text = lambda x: re.sub(r'^\d+\.\s*', " ", x).strip()

In [ ]:
df_full["text"] = df_full["text"].apply(clean_text)

In [ ]:
df_full.to_csv(os.path.join(store_path, "synthetic_data_full.csv"), index=False)

In [ ]:
# make train test split 6:2:2

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42, stratify=df_full["label"])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

len(train_df), len(test_df), len(val_df)

(600, 200, 200)

In [ ]:
train_df.to_csv(os.path.join(store_path, "train_data.csv"), index=False)
test_df.to_csv(os.path.join(store_path, "test_data.csv"), index=False)
val_df.to_csv(os.path.join(store_path, "val_data.csv"), index=False)

In [3]:
llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.8
        )

In [4]:
a =llm.invoke("""TASK 
You get list of short descriptions of a companies' business models. Use these descriptions to formulate it into a typical description within an annual report.

DEFINITION
This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. 



Here are some examples of descriptions of these classes: 
```
Example 1:
The company, together with its subsidiaries, operates as one of the largest vertically integrated agricultural groups in Ukraine, engaging in the production, storage, processing, and sale of agricultural products. The company's key activities include breeding pigs, processing pork, and producing wheat and sunflower. The Group focuses on three winter crops and two summer crops, maintaining a crop ratio of 60% winter crops (wheat, barley, rapeseed) and 40% summer crops (sunflower, corn). This strategy is designed to ensure a steady supply of basic food products, which remain in high demand, especially during wartime. The Group has also secured additional financing to cover key production costs and maintain strategic reserves of essential supplies.

Example 2:
The company initially focused on the business activities of catching skipjack and red snapper with a target of export sales. Over time, in 1983, the Company started its first production operations which was marked by the establishment of its factory in Kendari, Southeast Sulawesi.Furthermore, in order to expand its market share, the Company expanded into the integrated fish processing industry which includes processing activities. Since then, the Company has been able to produce processed marine products that contain high protein and added value, such as fish filets, tuna, octopus, squid, and other value-added products.In managing its business, the Company is committed to always consider sustainability values in all aspects. Not merely focusing on financial performance, the Company also promotes business alignment and harmonization as well as provides optimal benefits to stakeholders. The Company believes that by establishing a mutually beneficial business ecosystem, long-term business continuity will be maintained.

Example 3:
The company is the global leader in the full cycle breeding, production and sale of Yellowtail Kingfish and is renowned world-wide for its exceptionally high quality fish. Our company is recognised for innovation and its high degree of expertise in the farming of Yellowtail Kingfish. We are the largest producer of aquaculture Yellowtail Kingfish outside of Japan. Our diverse customer base has long appreciated the consistently high quality of our fish and our reliability in supplying our fresh and frozen range to markets all over the world 52 weeks of the year.
```

BUSINESS MODELS
0: A tech platform connects local wood suppliers with artisans looking for raw materials, streamlining the procurement process.
1: A company that offers consulting services to help landowners manage their forests for both timber and non-timber product extraction.
2: An organic dairy farm producing artisanal cheeses and yogurts, paired with a farm-to-table café on-site.
3: A cooperative that aggregates products from various mixed farms to sell under a single organic brand.
4: A sustainable sea cucumber farm that caters to niche markets in Asian health and culinary sectors.
5: An eco-friendly farm that integrates permaculture principles, offering educational courses and a variety of organic seasonal produce.

INSTRUCTIONS
 - For each given Business Model, create a business model description that is in a similar format as the examples
 - Do NOT exactly copy phrases, sentence patterns, or structure from the examples
 - Think of the examples as constraints, not templates
 - Also use terms other than those used in the sector definition to make the examples more diverse""")

In [5]:
print(a.content)

### Business Model Descriptions

**Business Model 0:**
The company operates a digital platform designed to facilitate connections between local timber suppliers and artisans in need of raw materials. By streamlining the procurement process, the platform enhances efficiency and transparency in the sourcing of wood products. This innovative approach not only supports local suppliers by giving them access to a broader market but also empowers artisans to find sustainable materials that meet their specific project requirements. The company is committed to promoting responsible sourcing practices, ensuring that both artisans and suppliers contribute to a more sustainable timber economy.

**Business Model 1:**
The company specializes in providing expert consultancy services to landowners seeking to optimize the management of their forested areas. With a focus on sustainable practices, the company assists clients in extracting both timber and non-timber products while maintaining ecological b

In [280]:
b = llm.invoke("""TASK 
You get list of short descriptions of a companies' business models. Use these descriptions to formulate it into a typical description within an annual report.

DEFINITION
This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. 



Here are some examples of descriptions of these classes: 
```
Example 1:
The company, together with its subsidiaries, operates as one of the largest vertically integrated agricultural groups in Ukraine, engaging in the production, storage, processing, and sale of agricultural products. The company's key activities include breeding pigs, processing pork, and producing wheat and sunflower. The Group focuses on three winter crops and two summer crops, maintaining a crop ratio of 60% winter crops (wheat, barley, rapeseed) and 40% summer crops (sunflower, corn). This strategy is designed to ensure a steady supply of basic food products, which remain in high demand, especially during wartime. The Group has also secured additional financing to cover key production costs and maintain strategic reserves of essential supplies.

Example 2:
The company initially focused on the business activities of catching skipjack and red snapper with a target of export sales. Over time, in 1983, the Company started its first production operations which was marked by the establishment of its factory in Kendari, Southeast Sulawesi.Furthermore, in order to expand its market share, the Company expanded into the integrated fish processing industry which includes processing activities. Since then, the Company has been able to produce processed marine products that contain high protein and added value, such as fish filets, tuna, octopus, squid, and other value-added products.In managing its business, the Company is committed to always consider sustainability values in all aspects. Not merely focusing on financial performance, the Company also promotes business alignment and harmonization as well as provides optimal benefits to stakeholders. The Company believes that by establishing a mutually beneficial business ecosystem, long-term business continuity will be maintained.

Example 3:
The company is the global leader in the full cycle breeding, production and sale of Yellowtail Kingfish and is renowned world-wide for its exceptionally high quality fish. Our company is recognised for innovation and its high degree of expertise in the farming of Yellowtail Kingfish. We are the largest producer of aquaculture Yellowtail Kingfish outside of Japan. Our diverse customer base has long appreciated the consistently high quality of our fish and our reliability in supplying our fresh and frozen range to markets all over the world 52 weeks of the year.
```

BUSINESS MODELS
0: A tech platform connects local wood suppliers with artisans looking for raw materials, streamlining the procurement process.
1: A company that offers consulting services to help landowners manage their forests for both timber and non-timber product extraction.
2: An organic dairy farm producing artisanal cheeses and yogurts, paired with a farm-to-table café on-site.
3: A cooperative that aggregates products from various mixed farms to sell under a single organic brand.
4: A sustainable sea cucumber farm that caters to niche markets in Asian health and culinary sectors.
5: An eco-friendly farm that integrates permaculture principles, offering educational courses and a variety of organic seasonal produce.
6: A mixed operation that incorporates agro-tourism, allowing visitors to experience crop harvesting while selling farm products on-site.
7: A forestry management consultancy that assists landowners in optimizing timber yield while maintaining ecological health in their forests.
8: A boutique retail shop that curates and sells handcrafted goods made from roundwood and wild forest products.
9: An organic vineyard that not only produces wine but also operates a farm-to-glass restaurant, providing farm tours and tastings.
10: A timber construction company specializing in homes built from locally sourced roundwood, emphasizing energy efficiency.
11: A community-supported fishery (CSF) that offers subscription boxes of seasonal, wild-caught fish directly from local fishermen to households.
12: A multi-generational family business that has been producing firewood and charcoal using traditional methods for decades.
13: A farm specializing in heirloom vegetables, providing seed shares to local gardeners and running seasonal pop-up markets.
14: A mixed farming operation that grows vegetables alongside specialty mushrooms, supplying local chefs with unique ingredients.
15: An online marketplace for niche forest products, connecting small producers of non-timber goods with a broader audience.
16: A permaculture farm that integrates tree crops, vegetables, and livestock to create a self-sustaining ecosystem.
17: An agribusiness that raises goats for milk and cheese production, selling to local markets and hosting farm visits for educational purposes.
18: A specialty farm that produces organic spices and herbs, targeting gourmet chefs and health food stores with unique offerings.
19: A cooperative that brings together mushroom growers and crop farmers to enhance biodiversity and offer unique farm products.

INSTRUCTIONS
 - For each given Business Model, create a business model description that is in a similar format as the examples
 - Do NOT exactly copy phrases, sentence patterns, or structure from the examples
 - Think of the examples as constraints, not templates
 - Also use terms other than those used in the sector definition to make the examples more diverse""")

In [281]:
print(b.content)

### Business Model Descriptions

**Business Model 0:**  
The company operates a cutting-edge digital platform that serves as a bridge between local timber suppliers and artisans in need of raw materials. By streamlining the procurement process, the platform enhances efficiency and fosters direct connections among stakeholders, ensuring that artisans can access quality wood sustainably sourced from nearby regions. This innovative approach not only supports local economies but promotes responsible forestry practices.

**Business Model 1:**  
This organization specializes in providing expert consulting services to landowners, focusing on sustainable forest management practices. The firm aids clients in maximizing both timber and non-timber product yields, while prioritizing ecological preservation. Through tailored management plans and ongoing support, the company empowers landowners to achieve economic benefits alongside environmental stewardship.

**Business Model 2:**  
Located on a pi